# Lekcja 6 — Klasyczny odbiornik (baseline)

## Cel
Zbudować **Classical Baseline** — krzywa **BLER vs SNR** do porównania z CNN.

## Łańcuch odbiornika
```
RX Grid → LS Channel Estimation → LMMSE Equalization → Demapper → LLR → LDPC
```

## Dlaczego to kluczowe?
Bez baseline wynik „BLER = 0.03” nic nie znaczy. Potrzebujesz:
```
SNR = 5 dB → Classical BLER = 0.08, Neural BLER = 0.04  ✓
```

## Co dokładnie zastąpi sieć neuronowa?
| Blok klasyczny | Rola | Zastąpiony przez CNN? |
|----------------|------|----------------------|
| Channel Estimation (LS) | $\hat{H}$ z pilotów | **TAK** |
| LMMSE Equalizer | usuwa ISI/fading | **TAK** |
| Demapper | symbole → LLR | **TAK** |
| LDPC Decoder | LLR → bits | **NIE** |


In [ ]:
import sys
from pathlib import Path

# Dodaj src/ do PYTHONPATH
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import sionna as sn
    import sionna.phy
except ImportError as e:
    raise ImportError(
        "Brak Sionny. Uruchom z katalogu magisterka/: ./scripts/drun sync"
    ) from e

from src.utils.setup import print_environment, get_device

sn.phy.config.seed = 42
device = get_device()
print_environment()


## End-to-end classical link (OFDM + TDL + LDPC)

In [ ]:
from sionna.phy.ofdm import (
    ResourceGrid,
    ResourceGridMapper,
    LSChannelEstimator,
    LMMSEEqualizer,
)
from sionna.phy.channel import OFDMChannel
from sionna.phy.channel.tr38901 import TDL
from sionna.phy.mimo import StreamManagement

# Parametry linku
NUM_BPS = 2
K = 256
N = 512
coderate = K / N

encoder = sn.phy.fec.ldpc.LDPC5GEncoder(K, N)
decoder = sn.phy.fec.ldpc.LDPC5GDecoder(encoder, hard_out=True)
constellation = sn.phy.mapping.Constellation("qam", NUM_BPS)
mapper = sn.phy.mapping.Mapper(constellation=constellation)
demapper = sn.phy.mapping.Demapper("app", constellation=constellation)

rg = ResourceGrid(
    num_ofdm_symbols=14,
    fft_size=64,
    subcarrier_spacing=30e3,
    cyclic_prefix_length=6,
    pilot_pattern="kronecker",
    pilot_ofdm_symbol_indices=[2, 11],
)
rg_mapper = ResourceGridMapper(rg)

tdl = TDL(model="A", delay_spread=30e-9, carrier_frequency=3.5e9)
ofdm_ch = OFDMChannel(tdl, rg, add_awgn=True, normalize_channel=True, return_channel=True)

stream_mgmt = StreamManagement([[0]], 1)
ch_est = LSChannelEstimator(rg, interpolation_type="nn")
equalizer = LMMSEEqualizer(rg, stream_mgmt)

print("Gotowe bloki klasycznego odbiornika.")


In [ ]:
def classical_transmit_receive(batch_size: int, ebno_db: float):
    no = sn.phy.utils.ebnodb2no(ebno_db, NUM_BPS, coderate)
    bits = sn.phy.mapping.BinarySource()([batch_size, K])
    cw = encoder(bits)
    x_sym = mapper(cw)
    x_grid = rg_mapper(x_sym.reshape(batch_size, 1, 1, -1))

    # Kanał OFDM (freq domain) — zwraca odebrany resource grid
    y_grid, _ = ofdm_ch(x_grid, no)

    # Klasyczny odbiornik: estymacja → equalizacja → demapping → LDPC
    h_hat, err_var = ch_est(y_grid, no)
    x_hat, _ = equalizer(y_grid, h_hat, err_var, no)
    llr = demapper(x_hat.reshape(batch_size, -1), no)
    bits_hat = decoder(llr)
    return bits, bits_hat

# Szybki test
b, bh = classical_transmit_receive(32, ebno_db=5.0)
from src.utils.metrics import bler
print("BLER @ 5 dB:", bler(b, bh))


## Symulacja BLER vs SNR

In [ ]:
snr_db = np.arange(0, 11, 2)
bler_vals = []

for ebno in snr_db:
    b, bh = classical_transmit_receive(64, ebno)
    bler_vals.append(bler(b, bh))

plt.figure(figsize=(7, 5))
plt.semilogy(snr_db, bler_vals, "o-", label="Classical (LS + LMMSE + LDPC)")
plt.xlabel("Eb/N0 [dB]")
plt.ylabel("BLER")
plt.grid(True, which="both")
plt.legend()
plt.title("Baseline — zapisz tę krzywą do porównania z CNN")
plt.show()


## Podsumowanie
To jest Twój **Experiment 1 — Classical baseline**.

## Ćwiczenie
Zapisz wyniki `snr_db` i `bler_vals` do pliku `experiments/classical_bler.npy`.

**Następna lekcja:** `07_neural_receiver.ipynb`
